<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); padding: 40px 30px; border-radius: 12px; text-align: center; margin-bottom: 10px;">
  <div style="font-size: 64px; margin-bottom: 10px;">🎵</div>
  <h1 style="color: #e94560; font-size: 2.2em; margin: 0 0 8px 0; font-family: Georgia, serif;">Investigación de datos</h1>
  <h2 style="color: #a8dadc; font-size: 1.4em; margin: 0 0 20px 0; font-weight: normal;">Caso Chinook · Tienda de música digital</h2>
  <p style="color: #ccc; font-size: 0.95em; max-width: 500px; margin: 0 auto;">SQL + Python · Análisis exploratorio · Visualización</p>
</div>

## ¿Qué es Chinook?

**Chinook** es una base de datos de ejemplo de una tienda de música digital.
Incluye tablas conectadas entre sí como:

- `Customer` (clientes)
- `Invoice` (facturas)
- `InvoiceLine` (detalle de cada factura)
- `Track` (canciones)
- `Genre` (géneros musicales)

🎯 En este cuaderno vais a trabajar como analistas: formular preguntas, consultar datos y justificar conclusiones con evidencia.

---


# 🔍 Investigación: ¿quién mueve más dinero en Chinook?

Ya sabéis hacer consultas en DB Browser. Ahora el reto es conectar la base de datos desde Python y responder **4 preguntas de negocio** usando `pandas` + `sqlite3`.

No hay una solución única. Si tu consulta funciona y el resultado tiene sentido: está bien.

---

**Normas del reto:**
- Una celda por pregunta con tu consulta SQL dentro de `pd.read_sql(...)`
- El resultado visible (que se vea la tabla)
- **Una línea de texto** debajo explicando qué significa lo que ves
- En cada pregunta, haz primero una versión de prueba (por ejemplo con `LIMIT 5`) y luego la versión final

Al cerrar, revisaremos las soluciones **en conjunto en formato fórum** 👇

## Paso 0 · Conexión a la base de datos

Ejecuta esta celda primero. Si ves una tabla con nombres de tablas, estás dentro ✅

In [20]:
import sqlite3
import pandas as pd

# Ajusta la ruta si tu Chinook.db está en otra carpeta
con = sqlite3.connect("Chinook.db")

# ¿Qué tablas hay disponibles?
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con)

,name
0,Album
1,Artist
2,Customer
3,Employee
4,Genre
5,Invoice
6,InvoiceLine
7,MediaType
8,Playlist
9,PlaylistTrack


---

## Pregunta 0 (calentamiento) — ¿Cuántas facturas hay en total?

Antes de empezar las 4 preguntas, valida que todo funciona con una consulta muy corta.

Si esta sale bien, ya tienes conexión y sintaxis controladas ✅

In [21]:
# Pregunta 0: total de facturas
pd.read_sql("""
    SELECT COUNT(*) AS total_facturas
    FROM Invoice
""", con)

,total_facturas
0,412


---

## Pregunta 1 — ¿Qué empleado tiene más clientes asignados?

> 💡 Pista: hay una columna en la tabla `Customer` que apunta a la tabla `Employee`. ¿Cuál es?

Escribe tu consulta en la celda de abajo y muestra el resultado.

Cuando lo tengas, añade una línea en texto explicando qué significa el número que ves.

In [22]:
# Pregunta 1: ¿Qué empleado tiene más clientes asignados?
# Escribe tu consulta aquí:
pd.read_sql(
"""
SELECT 
    e.FirstName || ' ' || e.LastName AS Nombre_Empleado, 
    COUNT(c.CustomerId) AS total_clientes
FROM Employee e
JOIN Customer c ON e.EmployeeId = c.SupportRepId
GROUP BY e.EmployeeId
ORDER BY total_clientes DESC;
"""

, con)

,Nombre_Empleado,total_clientes
0,Jane Peacock,21
1,Margaret Park,20
2,Steve Johnson,18


**Mi interpretación:** *(escribe aquí en una frase qué significa el resultado)*

---

## Pregunta 2 — ¿En qué mes del año se factura más? ¿Y menos?

> 💡 Pista: la fecha de factura está guardada como texto. Puedes extraer el mes con `strftime('%m', InvoiceDate)`.

Cuando lo tengas, añade una línea explicando si el patrón tiene sentido o te parece sorprendente.

In [23]:
# Pregunta 2: ¿En qué mes se factura más? ¿Y menos?
# Escribe tu consulta aquí:

pd.read_sql("""
    -- TU CONSULTA AQUÍ
 SELECT 
strftime('%m', InvoiceDate) , 
SUM(Total)
FROM Invoice
GROUP BY strftime('%m', InvoiceDate)
ORDER BY sum(total) DESC;  
            """, con)

,"strftime('%m', InvoiceDate)",SUM(Total)
0,01,201.12
1,06,201.10
2,04,198.14
3,08,198.10
4,09,196.20
5,03,195.10
6,10,193.10
7,05,193.10
8,07,190.10
9,12,189.10


**Mi interpretación:** *(escribe aquí en una frase qué significa el resultado)*

---

## Pregunta 3 — ¿Hay clientes que hayan comprado solo una vez? ¿Cuántos son?

> 💡 Pista: una compra = una factura en la tabla `Invoice`. Agrupa por cliente y filtra los que solo tienen 1.

¿Sorprende la cantidad? Escribe tu conclusión debajo del resultado.

In [24]:
# Pregunta 3: ¿Cuántos clientes han comprado solo una vez?
# Escribe tu consulta aquí:

pd.read_sql("""
    SELECT CustomerId , count(InvoiceId) as total_compras
FROM Invoice
GROUP by CustomerId
HAVING count(total) =1;
""", con)

,CustomerId,total_compras


**Mi interpretación:** *(escribe aquí en una frase qué significa el resultado)*

---

## Pregunta 4 — ¿Cuál es el género musical más vendido por tracks comprados?

> 💡 Esta la haremos en 2 pasos:
> 1) Une `InvoiceLine` con `Track` para comprobar que puedes llegar a `TrackId` y `GenreId`.
> 2) Añade `Genre`, agrupa y ordena para obtener el top.

Esta es la más difícil. Si llegas aquí, enhorabuena 🎉

In [ ]:
# Pregunta 4: ¿Cuál es el género más vendido por número de tracks comprados?

# Paso 1 (prueba): valida el cruce InvoiceLine + Track
pd.read_sql("""
SELECT *,
SUM(Quantity) AS "Numero canciones" 
FROM InvoiceLine il
JOIN TRACK t ON il.TrackId = t.TrackId 
JOIN Genre g on t.GenreId = g.GenreId 
GROUP BY g.GenreId 
ORDER BY "Numero canciones" DESC Limit 5;
""", con)



,InvoiceLineId,InvoiceId,TrackId,UnitPrice,Quantity,TrackId,Name,AlbumId,MediaTypeId,GenreId,Composer,Milliseconds,Bytes,UnitPrice,GenreId,Name,Numero Canciones
0,1,1,2,0.99,1,2,Balls to the Wall,2,2,1,None,342562,5510424,0.99,1,Rock,835
1,34,5,207,0.99,1,207,Meditação,21,1,7,Tom Jobim - Newton Mendoça,148793,4865597,0.99,7,Latin,386
2,19,4,78,0.99,1,78,Master Of Puppets,9,1,3,Apocalyptica,436453,14375310,0.99,3,Metal,264
3,22,5,99,0.99,1,99,Your Time Has Come,11,1,4,"Cornell, Commerford, Morello, Wilk",255529,8273592,0.99,4,Alternative & Punk,244
4,17,4,66,0.99,1,66,Por Causa De Você,8,1,2,None,169900,5536496,0.99,2,Jazz,80


**Mi interpretación:** *(escribe aquí en una frase qué significa el resultado)*

---

## Cierre en fórum (sin entrega)

- No hay entrega individual hoy
- Guarda tus consultas para poder enseñarlas si te toca compartir
- Cerraremos la sesión resolviendo dudas y comparando enfoques **en conjunto**
- Piensa qué pregunta te costó más y por qué, para comentarla en el fórum